# Cross-Basin Generalisation for Tropical Cyclone Forecasting using TropiCycloneNet

**Team:** Loïc Bouxirot, Yasmin Akhmedova, Samuel Zhang

**Module:** ELEC70127 - ML for Tackling Climate Change

**Date:** March 2026

## Introduction

Tropical cyclone (TC) forecasting in data-scarce ocean basins is limited by small training sets. Operational models require extensive historical records that simply do not exist in many regions [1]. Transfer learning from data-rich to data-scarce basins is a possible solution, and there is preliminary evidence this works: Qu et al. (2025) fine-tuned a Western North Pacific-trained transformer on the Eastern North Pacific and outperformed operational forecasting systems [7]. This project investigates whether deep learning models trained on the **Western Pacific (WP)**, one of the most active and well-observed basins, can generalise to the **South Pacific (SP)**, a data-scarce basin in the opposite hemisphere.

The WP to SP transfer is particularly challenging because the hemisphere flip introduces a **Coriolis reversal**: WP storms predominantly track NW/W, while SP storms track SE/W/SW [1]. A naive WP-trained model will systematically mispredict SP storm directions. Huang et al. (2025) show that multi-basin models achieve 61-73% better track prediction than WP-only models on Southern Hemisphere basins [1]. However, the underlying atmospheric physics, specifically sea surface temperature (SST) and vertical wind shear, is shared across both basins [2]. Our exploratory analysis confirms this: as Figure 1 shows, WP and SP have substantially overlapping SST and wind shear distributions. **This suggests that physics-informed features should transfer even if direction labels do not.** `[REVIEW: ensure the conclusion addresses whether this hypothesis held up]`

_Figure 1: SST and vertical wind shear distributions across all six TCND basins. WP (brown) and SP (orange) overlap substantially on both variables, supporting the hypothesis that intensity-related physics is shared across hemispheres._

![SST and wind shear distributions by basin](figures/sst_shear_by_basin.png)

### What we explore

| Question | Approach | Key techniques |
| --- | --- | --- |
| **Can a U-Net (local spatial features) transfer across basins?** | Baseline U-Net, then an improved variant with temporal conditioning | - **SE attention** lets the model learn which input channels (e.g. SST, wind) matter most <br> - **Augmentation** (Mixup, CutOut, noise) artificially increases training variety to reduce overfitting on our small dataset <br> - **FiLM conditioning** [3] tells the model *when* in a storm's lifecycle each sample occurs, so it can adjust predictions accordingly |
| **Can an FNO (global spectral features) transfer across basins?** | Baseline FNO, then an improved v2 with padding and temporal conditioning | - The FNO works in the **frequency domain**: instead of scanning local patches like a U-Net, it learns large-scale atmospheric patterns using Fourier transforms [4] <br> - v2 adds **reflect padding** to fix edge artifacts <br> - v2 adds **FiLM conditioning** for temporal awareness |
| **Can a hybrid (U-FNO) combine the best of both?** | Gated architecture that blends U-Net and FNO branches | - Each layer has three parallel paths (spectral, spatial, residual) <br> - **Learnable gate weights** [5] let the model discover the best mix of local and global features at each depth |

Each model performs **dual-head classification** at every 6-hourly timestep:

- **Direction** (8 classes): E, SE, S, SW, W, NW, N, NE, representing the compass direction the storm will move in the next 24 hours.
- **Intensity change** (4 classes): Weakening, Steady, Slow-intensification, Rapid-intensification.

Our three-stage evaluation pipeline consists of:
1. **Zero-shot transfer**: train on WP, apply hemisphere-aware preprocessing, evaluate on SP with no fine-tuning.
2. **Fine-tuning**: fine-tune on a small SP training set (12 storms, ~354 samples) and re-evaluate.
3. **Transfer gap analysis**: quantify the performance drop from in-basin to cross-basin and how much fine-tuning recovers.

> **Note on ResNet-152:** We also implemented a ResNet-152 baseline (`supplementary-notebooks/models/resnet.ipynb`), but it was massively overparameterised for our dataset (58.7M parameters for ~3,250 samples) and had the largest transfer gap on direction. Our original proposal included five architectures (U-Net, ResNet, FNO, PI-GAN, Transformer); following supervisor feedback, we narrowed the focus to U-Net and FNO as the two most promising families, adding temporal conditioning (FiLM) and a hybrid (U-FNO) to explore improvements within those families.

## Part 1: Data & Preprocessing

We use the **TropiCycloneNet Dataset (TCND)** [1], a multimodal tropical cyclone dataset covering 1950–2023 across six ocean basins. Our preprocessing pipeline (`supplementary-notebooks/preprocessing/data-preprocessing-pipeline.ipynb`) transforms raw TCND data into model-ready tensors.

### 1.1 Dataset Overview

Three modalities are available per 6-hourly timestep:

_Table 1: Data Modalities_

| # | Modality | Format | Content |
| --- | --- | --- | --- |
| 1 | **Data3D** | NetCDF (.nc) | - 81×81 gridded atmospheric fields <br> - SST (1 channel) <br> - u/v wind at 4 pressure levels (200, 500, 850, 925 hPa) <br> - Geopotential height at 4 pressure levels <br> - = 13 base channels total |
| 2 | **Env-Data** | NumPy (.npy) | - 94-dim structured environmental features per timestep <br> - Includes pre-computed 24h direction and intensity-change labels <br> - After preprocessing, reduced to 40 physics-relevant features (see Section 1.2) |
| 3 | **Data1D** | TSV (.txt) | - Per-storm track files <br> - lat/lon offsets, normalised wind and pressure <br> - 4 features retained |

### WP vs SP Comparison

We train on WP and test on SP. The table below highlights two things: (1) WP is much larger than SP, giving us a data-rich source domain, and (2) while the storm directions are mirrored between hemispheres, the underlying physics is similar, which is why cross-basin transfer is worth attempting.

_Table 2: Basin Comparison_

|  | WP (train) | SP (test) | What this tells us |
| --- | --- | --- | --- |
| Storms | 131 | 30 | WP has 4.4x more storms, making it a strong source domain for transfer learning |
| Timesteps | 4,562 | 922 | SP is data-scarce, which is exactly the problem we are trying to solve |
| Top directions | NW (30%), W (28%) | SE (25%), W (19%) | Directions are mirrored across hemispheres due to Coriolis reversal. This is why naive transfer fails on direction. |
| Physics (SST, wind shear) | See Figure 1 | See Figure 1 | Both variables overlap substantially between basins (Figure 1), supporting cross-basin transfer of intensity-related features |

**Key insight:** The physics that governs storm *intensity* (SST, wind shear) is shared across both basins, but storm *direction* is hemisphere-mirrored. This means a WP-trained model's intensity features should transfer well to SP, but its direction predictions will be systematically wrong unless we apply hemisphere-aware preprocessing (Section 1.2).

### 1.2 Preprocessing Pipeline

The preprocessing pipeline applies four key transformations. Full implementation is in `supplementary-notebooks/preprocessing/data-preprocessing-pipeline.ipynb`.

**1. Basin-leaking feature removal**
*Why:* Some features in the raw data directly reveal which ocean basin a storm belongs to. If we leave these in, the model can take shortcuts (e.g. "this is WP, so predict NW") instead of learning generalisable physics. We need to force the model to rely on atmospheric patterns that work across basins.
*What:* 54 of the 94 environmental dimensions encode basin identity (area one-hot, longitude/latitude bins, raw coordinates). We strip these, retaining only the 40 physics-relevant dimensions: wind, move velocity, intensity class, month, and historical direction/intensity labels.

**2. Hemisphere-aware transforms (SP only)**
*Why:* WP is in the Northern Hemisphere, SP is in the Southern Hemisphere. Storms rotate in opposite directions due to the Coriolis effect, so a "northwestward" storm in WP is physically analogous to a "southwestward" storm in SP. Without correcting for this, the model would see completely different direction patterns and fail to transfer.
*What:* Direction labels are reflected along the N/S axis so that "poleward" maps to the same class in both hemispheres. Meridional wind (v) is flipped so that poleward flow has a consistent sign.

**3. Derived physics channels**
*Why:* The raw grid data contains wind speed and geopotential height at different altitudes, but the model has to figure out on its own that the *difference* between upper and lower winds (shear) and the *rotation* of wind (vorticity) matter for cyclone behaviour. By computing these explicitly, we give the model direct access to known physical drivers of storm intensity.
*What:* Two channels are appended to the 13 base grid channels, giving **15 total**:
- Wind shear magnitude: $\sqrt{(u_{200} - u_{850})^2 + (v_{200} - v_{850})^2}$
- 850 hPa relative vorticity: $\frac{\partial v_{850}}{\partial x} - \frac{\partial u_{850}}{\partial y}$ (centred finite differences)

**4. Normalisation**
*Why:* Different atmospheric variables have very different scales (e.g. SST in Kelvin ~300, wind in m/s ~10, geopotential in m²/s² ~50,000). Without normalisation, the model would be dominated by whichever variable has the largest numbers. We also compute statistics only from WP training data to avoid leaking any information about SP into the model before evaluation.
*What:* Per-channel z-score computed exclusively from WP training data and applied to all splits.

**Missing Data1D handling:** If a timestep has no matching Data1D entry, the entire timestep is dropped from the dataset (grid, env, and labels are all excluded). We do not zero-fill because zero wind speed or zero pressure has a real physical meaning and would feed the model false information. Drop counts are logged per storm and per split.

### 1.3 Data Reduction & Final Split Sizes

The preprocessing pipeline substantially reduces the raw dataset. The main source of data loss is missing Data1D entries for WP storms from 2020–2021, which forces entire timesteps to be dropped (we do not zero-fill, as explained above).

_Table 3a: Data Reduction Summary_

| Stage | Storms | Timesteps | Notes |
| --- | --- | --- | --- |
| **Raw TCND (WP + SP)** | 161 | 5,484 | WP: 131 storms / 4,562 ts; SP: 30 storms / 922 ts |
| **After Data1D drops** | 117 | 3,621 | 27% fewer storms, 34% fewer timesteps |
| **Valid labels (24h-ahead)** | — | 3,153 (87%) | First few timesteps per storm have no 24h-ahead target |

_Table 3b: Final Split Sizes_

| Split | Storms | Timesteps | Valid Labels | Purpose |
| --- | --- | --- | --- | --- |
| `wp_train` | 69 | 2,092 | 1,816 | Source-domain training |
| `wp_val` | 18 | 607 | 535 | Source-domain validation |
| `sp_test` | 15 | 427 | 367 | Zero-shot transfer evaluation |
| `sp_ft_train` | 12 | 402 | 354 | Target-domain fine-tuning |
| `sp_ft_val` | 3 | 93 | 81 | Fine-tuning validation (very small) |
| **Total** | **117** | **3,621** | **3,153** | |

**Key takeaway:** The effective training set is ~1,816 valid WP samples. This data-limited regime strongly favours architectures with good inductive biases and aggressive regularisation over raw model capacity — a theme that recurs throughout our results.

### 1.5 Temporal Feature Extraction

The preprocessing pipeline described above produces spatial (grid), environmental, and track features — but none of these tell the model *when* a timestep occurs within a storm's lifecycle, what time of day it is, or what season it is. Following supervisor feedback (March 18 meeting), we added a temporal feature extraction step that computes a 6-dimensional vector per timestep to address this.

_Table 4: Temporal Features_

| Index | Feature | Formula | What it captures |
|-------|---------|---------|-----------------|
| 0 | `storm_progress` | $i / (N_t - 1)$ | Position within the storm lifecycle (0 = genesis, 1 = dissipation) |
| 1 | `hour_sin` | $\sin(2\pi \cdot h / 24)$ | Time of day — sin component |
| 2 | `hour_cos` | $\cos(2\pi \cdot h / 24)$ | Time of day — cos component |
| 3 | `month_sin` | $\sin(2\pi \cdot (m-1) / 12)$ | Season — sin component |
| 4 | `month_cos` | $\cos(2\pi \cdot (m-1) / 12)$ | Season — cos component |
| 5 | `storm_duration_norm` | $N_t / N_{\max}$ | Storm longevity relative to the longest storm in the dataset |

**Design choices:**

- **Cyclical encoding** (features 1–4): Raw hour or month values would imply that hour 23 and hour 0, or December and January, are far apart. Sin/cos encoding places them adjacent on a unit circle. Both components are needed because sin alone is ambiguous ($\sin$ is the same for hour 6 and hour 18).
- **Progress vs duration** (features 0 and 5): These capture complementary information. `storm_progress` varies *within* a storm (where are we in the lifecycle?), while `storm_duration_norm` is constant *within* a storm but varies *across* storms (how long-lived is this storm overall?). Longer storms tend to be more intense, so duration gives the model a useful structural signal.

These features are injected into models via **FiLM conditioning** (see Part 2), enabling each convolutional block to adapt its processing based on temporal context.

The temporal extraction pipeline also replicates the Data1D filtering from Section 1.3, ensuring the output `.pt` files align exactly with the other modalities (117 storms, 3,621 timesteps).

_Full implementation: `supplementary-notebooks/preprocessing/temporal-feature-extraction.ipynb`_

In [ ]:
# Temporal feature computation per storm
# (excerpt from supplementary-notebooks/preprocessing/temporal-feature-extraction.ipynb)

def build_time_features(group_df, max_storm_len, n_features=6) -> torch.Tensor:
    """Build (N_t, 6) tensor for one storm from its sorted rows."""
    n = len(group_df)
    feats = torch.zeros(n, n_features)
    duration_norm = n / max(max_storm_len, 1)

    for i, (_, row) in enumerate(group_df.iterrows()):
        s = str(row["datetime"])
        month, hour = int(s[4:6]), int(s[8:10])
        progress = i / max(n - 1, 1)

        feats[i] = torch.tensor([
            progress,                                    # storm_progress (0→1)
            math.sin(TWO_PI * hour / 24.0),              # hour_sin
            math.cos(TWO_PI * hour / 24.0),              # hour_cos
            math.sin(TWO_PI * (month - 1) / 12.0),       # month_sin
            math.cos(TWO_PI * (month - 1) / 12.0),       # month_cos
            duration_norm,                               # storm_duration_norm
        ])

    return feats

**Interpretation:** Since observations are 6-hourly, the hour encoding produces exactly 4 distinct positions (00, 06, 12, 18 UTC) on the unit circle. The temporal feature extraction notebook verifies correctness with distribution histograms, polar scatter plots, and cross-checks against the label files to confirm alignment.

_See: `supplementary-notebooks/preprocessing/temporal-feature-extraction.ipynb`_

## Part 2: U-Net Architectures

The U-Net family proved to be the strongest architecture for cross-basin direction forecasting. We present the baseline followed by the temporally-conditioned variant.

### 2.1 Baseline U-Net

The baseline U-Net uses a 4-level encoder-decoder (32→64→128→256, bottleneck 512) with skip connections, residual blocks, SE channel attention, and DropPath regularisation. Features are fused via global average pooling of the decoder output (32-d) concatenated with environmental (40-d) and 1D track (4-d) inputs, giving a 76-d fused vector fed to two classification heads.

**Key design choices:**
- **SE attention**: learns per-channel importance, allowing the model to weight physically meaningful channels (SST, wind shear) over less informative ones
- **Aggressive augmentation**: Mixup (α=0.2), CutOut (2×16×16), Gaussian noise (σ=0.05), channel dropout (p=0.15), and EMA weights to combat overfitting on ~3,250 samples
- **HPO via Optuna** (30 trials × 30 epochs): found that the model is data-limited, not capacity-limited (base_channels=32 outperformed 64)

_Full implementation: `supplementary-notebooks/models/unet-baseline.ipynb`_

In [ ]:
# Squeeze-and-Excitation channel attention used in each encoder/decoder level
# (excerpt from supplementary-notebooks/models/unet-baseline.ipynb)

class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, max(ch // reduction, 4)),
            nn.GELU(),
            nn.Linear(max(ch // reduction, 4), ch),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1)
        return x * w

**Interpretation:** The SE block enables the U-Net to learn which atmospheric channels are most relevant for each prediction. This is particularly important for cross-basin transfer, where some channels carry universal physics (SST, wind shear) while others may encode basin-specific patterns. Encoder activation analysis showed that shallow features (level 0, 32 channels) contribute most, consistent with the data-limited regime.

_TODO: Insert figure — U-Net encoder activation heatmap showing channel importance across levels (`figures/unet_encoder_activations.png`)_

_TODO: Insert figure — U-Net training curves showing loss and accuracy over epochs (`figures/unet_learning_curves.png`)_

### 2.2 U-Net + FiLM (Temporal Conditioning)

Following supervisor feedback, we added **Feature-wise Linear Modulation (FiLM)** [3] to condition the U-Net on temporal features. FiLM applies a learned per-channel affine transform ($y = \gamma x + \beta$) derived from the time embedding, allowing the model to adapt its spatial feature processing based on storm lifecycle stage.

**Key additions over baseline:**
- **Time MLP**: `(6,) → 64 → 64` with GELU, producing a conditioning vector broadcast to all FiLM layers
- **FiLM injection**: After the second BatchNorm, before GELU activation, in every encoder and decoder block
- **Identity initialisation**: γ=1, β=0 so the model starts equivalent to the baseline. FiLM must "earn" its influence during training.

All other hyperparameters remain identical to the baseline (BASE_CHANNELS=32, N_LEVELS=4, EPOCHS=300, PATIENCE=50).

_Full implementation: `supplementary-notebooks/models/unet-film-temporal.ipynb` | Checkpoints: `unet_film_best_wp.pt`, `unet_film_best_ft.pt`_

In [ ]:
# Feature-wise Linear Modulation layer for temporal conditioning
# (excerpt from supplementary-notebooks/models/unet-film-temporal.ipynb)

class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation. Identity-init (gamma=1, beta=0)."""
    def __init__(self, cond_dim, channels):
        super().__init__()
        self.fc = nn.Linear(cond_dim, channels * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)
        self.fc.bias.data[:channels] = 1.0  # gamma=1

    def forward(self, x, cond):
        gamma, beta = self.fc(cond).chunk(2, dim=1)
        return gamma.unsqueeze(-1).unsqueeze(-1) * x + beta.unsqueeze(-1).unsqueeze(-1)

**Interpretation:** The identity initialisation is a deliberate design choice: it ensures the FiLM-conditioned model begins training from the same starting point as the baseline. If temporal features are informative, the model will learn non-trivial γ and β values during training; if not, it gracefully degrades to the baseline. This allows us to isolate the contribution of temporal awareness.

_TODO: Insert figure — U-Net + FiLM training curves compared with baseline (`figures/unet_finetune_curves.png`)_

---

## Part 3: FNO Architectures

The Fourier Neural Operator (FNO) family takes a fundamentally different approach: rather than learning local spatial patterns through convolutions, it learns global patterns in the **frequency domain** via spectral convolutions [4]. This is particularly appealing for cross-basin transfer because Fourier transforms are inherently mesh-invariant and resolution-invariant, as demonstrated by FourCastNet [8] for global weather forecasting.

### 3.1 Baseline FNO

The baseline FNO consists of a lifting layer → 2 spectral blocks (SpectralConv2d with truncated Fourier modes + skip connections + BatchNorm) → projection → global average pooling. Features are fused via the same late-fusion strategy as U-Net: pooled grid features concatenated with environmental and 1D inputs.

**Configuration:** `HIDDEN_CHANNELS=80, N_MODES=20, N_LAYERS=2`

**HPO:** Optuna (60 trials × 30 epochs). Best optimiser: Muon (fell back to AdamW for stability).

_Full implementation: `supplementary-notebooks/models/fno-baseline.ipynb`_

In [ ]:
# Fourier spectral convolution: the core FNO operation
# (excerpt from supplementary-notebooks/models/fno-baseline.ipynb)

class SpectralConv2d(nn.Module):
    """2D Fourier spectral convolution layer.
    Performs global convolution via FFT: learns complex-valued weights
    for the lowest (modes1 x modes2) Fourier coefficients."""

    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = (2 / (in_channels + out_channels)) ** 0.5
        self.weights1 = nn.Parameter(
            scale * (torch.rand(in_channels, out_channels, modes1, modes2,
                                dtype=torch.cfloat) - 0.5))
        self.weights2 = nn.Parameter(
            scale * (torch.rand(in_channels, out_channels, modes1, modes2,
                                dtype=torch.cfloat) - 0.5))

    def compl_mul2d(self, x, weights):
        """Complex-valued einsum: (B,Ci,H,W) x (Ci,Co,H,W) -> (B,Co,H,W)"""
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(B, self.out_channels, H, W // 2 + 1,
                             dtype=torch.cfloat, device=x.device)
        # Low-frequency modes (positive and negative frequencies)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(H, W))

**Interpretation:** The SpectralConv2d operates entirely in the frequency domain, learning which large-scale atmospheric patterns (low-frequency Fourier modes) are predictive of cyclone behaviour. Fourier mode analysis revealed that the baseline model predominantly learns **meridional (N-S) patterns**: the top active modes all have width-mode=0, suggesting large-scale atmospheric structure matters more than fine spatial details. However, the baseline suffered from two issues: **spectral leakage** at boundaries and **early overfitting** (stopped at epoch 18/80).

_TODO: Insert figure — Baseline FNO Fourier mode importance heatmap (`figures/fno_mode_importance.png`)_

_TODO: Insert figure — Baseline FNO training curves (`figures/fno_learning_curves.png`)_

### 3.2 FNO v2 (Padded Spectral Conv + FiLM)

FNO v2 addresses the baseline's limitations with three improvements, implemented after the supervisor meeting (see `README_implementation.md`):

1. **Spatial padding**: `F.pad(x, [9,9,9,9], mode='reflect')` before `rfft2`, then crop after `irfft2`. This reduces spectral leakage and Gibbs boundary artifacts that were polluting the Fourier signal.
2. **3 spectral layers** (baseline had 2): More capacity for learning fine-grained frequency patterns. FNO can exhibit "spectral leaps" (sudden accuracy jumps after long plateaus), so more layers + patience is warranted.
3. **FiLM conditioning**: Same mechanism as U-Net+FiLM, injected after BatchNorm in each spectral block.

**Configuration:** `HIDDEN_CHANNELS=32, N_MODES=12, N_LAYERS=3, PADDING=9, EPOCHS=150, PATIENCE=30`

_Full implementation: `supplementary-notebooks/models/fno-v2-padded-film.ipynb` | Checkpoints: `fno_v2_best_wp.pt`, `fno_v2_best_ft.pt`_

In [ ]:
# PaddedSpectralConv2d: key improvement in FNO v2
# (excerpt from supplementary-notebooks/models/fno-v2-padded-film.ipynb)

def forward(self, x):
    B, C, H, W = x.shape

    # Reflect-pad spatial dims to reduce boundary artifacts
    if self.padding > 0:
        x = F.pad(x, [self.padding, self.padding,
                      self.padding, self.padding], mode='reflect')

    Hp, Wp = x.shape[-2], x.shape[-1]
    x_ft = torch.fft.rfft2(x)

    out_ft = torch.zeros(B, self.out_channels, Hp, Wp // 2 + 1,
                         dtype=torch.cfloat, device=x.device)

    # Low-frequency modes (top-left and bottom-left corners)
    out_ft[:, :, :self.modes1, :self.modes2] = \
        self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
    out_ft[:, :, -self.modes1:, :self.modes2] = \
        self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

    x = torch.fft.irfft2(out_ft, s=(Hp, Wp))

    # Crop back to original spatial size
    if self.padding > 0:
        x = x[:, :, self.padding:-self.padding, self.padding:-self.padding]
    return x

**Interpretation:** The reflect padding is a simple but effective fix. Without it, the FFT assumes periodic boundary conditions: the right edge of the 81×81 grid wraps around to the left edge, creating discontinuities that manifest as spurious high-frequency components. Reflect padding smoothly extends the signal, giving the FFT a more accurate spectral representation.

_TODO: Insert figure — FNO v2 mode importance comparison with baseline (`figures/fno_v2_mode_importance.png`)_

_TODO: Insert figure — FNO v2 training curves (`figures/fno_v2_training_curves.png`)_

### 3.3 U-FNO (Hybrid Gated Architecture)

Inspired by Wen et al. (2022) [5], the U-FNO combines the strengths of both architecture families. Each `UFNOBlock` has three parallel branches fused with **learnable softmax-gated weights**:

_Table 5: U-FNO Branch Architecture_

| Branch | Role |
| --- | --- |
| `PaddedSpectralConv2d` | Global frequency-domain features (with reflect padding) |
| `UNetBranch` | Local spatial features (lightweight single-level encode-decode) |
| `nn.Conv2d(ch, ch, 1)` | Residual / identity pathway |

The gate weights are initialised uniformly (`torch.ones(3)/3`) and learned via softmax, allowing the model to discover which branch matters most at each layer.

**Configuration:** `HIDDEN_CH=32, N_MODES=12, N_LAYERS=3, PADDING=9, TIME_EMB_DIM=64`

_Full implementation: `supplementary-notebooks/models/ufno-hybrid.ipynb` | Checkpoints: `ufno_best_wp.pt`, `ufno_best_ft.pt`_

In [ ]:
# U-FNO block: 3 gated branches (spectral + unet + residual) + FiLM
# (excerpt from supplementary-notebooks/models/ufno-hybrid.ipynb)

class UFNOBlock(nn.Module):
    """U-FNO block: 3 gated branches (spectral + unet + residual) + FiLM."""
    def __init__(self, ch, modes, padding, time_emb_dim, dropout=0.1):
        super().__init__()
        self.spectral = SpectralConv2d(ch, ch, modes, modes, padding)
        self.unet     = UNetBranch(ch)
        self.residual = nn.Conv2d(ch, ch, 1)
        self.gate     = nn.Parameter(torch.ones(3) / 3)
        self.norm     = nn.BatchNorm2d(ch)
        self.film     = FiLMLayer(time_emb_dim, ch)
        self.dropout  = nn.Dropout2d(dropout)

    def forward(self, x, t_emb):
        g = F.softmax(self.gate, dim=0)
        out = g[0]*self.spectral(x) + g[1]*self.unet(x) + g[2]*self.residual(x)
        out = self.norm(out)
        if t_emb is not None:
            out = self.film(out, t_emb)
        out = self.dropout(F.gelu(out))
        return out + x  # residual connection

**Interpretation:** The gated architecture lets the model learn the optimal blend of global (spectral) and local (U-Net) features per layer. Gate weight visualisation reveals whether the model favours spectral or spatial processing at different depths, acting as a form of learned architecture search.

_TODO: Insert figure — U-FNO gate weight visualisation showing branch importance per layer_

---

## Part 4: Results & Analysis

We evaluate all models across the three-stage pipeline. Full comparison with visualisations is in `supplementary-notebooks/supplementary-analysis/model-comparison.ipynb`.

### 4.1 In-Basin Performance (WP Validation)

_Table 6: WP Validation Results_

| Metric | U-Net | U-Net + FiLM | FNO | FNO v2 | U-FNO |
| --- | --- | --- | --- | --- | --- |
| **Direction Accuracy** | **0.625** | — | 0.503 | — | — |
| Direction F1 (macro) | **0.447** | — | 0.364 | — | — |
| **Intensity Accuracy** | 0.510 | — | **0.558** | — | — |
| Intensity F1 (macro) | 0.433 | — | 0.410 | — | — |

> _Note: Cells marked "—" are pending results from the post-meeting models (U-Net+FiLM, FNO v2, U-FNO). These will be filled in once training completes. See `README_implementation.md` for training instructions._

**U-Net leads on direction** by a wide margin (+12pp over FNO). Its encoder-decoder architecture with skip connections preserves fine-grained spatial patterns that indicate storm movement direction. **FNO is marginally better on intensity**, likely because intensity change depends more on aggregate features (overall SST, shear magnitude) than spatial structure.

_TODO: Insert figure — Bar chart comparing accuracy across all models (`figures/comparison_accuracy_bars.png`)_

### 4.2 Zero-Shot Cross-Basin Transfer (SP Test)

_Table 7: Zero-Shot SP Results_

| Metric | U-Net | U-Net + FiLM | FNO | FNO v2 | U-FNO |
| --- | --- | --- | --- | --- | --- |
| **Direction Accuracy** | **0.414** | — | 0.248 | — | — |
| Direction F1 (macro) | **0.293** | — | 0.217 | — | — |
| **Intensity Accuracy** | 0.360 | — | 0.357 | — | — |
| Intensity F1 (macro) | 0.291 | — | 0.300 | — | — |

_Table 8: Transfer Gap (WP → SP)_

| Transfer Gap | U-Net | FNO |
| --- | --- | --- |
| Direction Accuracy | **−0.211** | −0.255 |
| Intensity Accuracy | **−0.150** | −0.201 |

All models suffer a substantial transfer gap (20–25pp on direction, 15–20pp on intensity). **U-Net retains the best zero-shot direction accuracy (0.414) and has the smallest transfer gap**, suggesting its learned spatial features generalise best across basins.

_TODO: Insert figure — Transfer gap waterfall chart (`figures/comparison_transfer_gap.png`)_

_TODO: Insert figure — Zero-shot confusion matrices for direction (`figures/comparison_cm_dir_zeroshot.png`)_

### 4.3 Fine-Tuned Performance (SP Test)

_Table 9: Fine-Tuned SP Results_

| Metric | U-Net | U-Net + FiLM | FNO | FNO v2 | U-FNO |
| --- | --- | --- | --- | --- | --- |
| **Direction Accuracy** | **0.401** | — | 0.346 | — | — |
| Direction F1 (macro) | **0.309** | — | 0.272 | — | — |
| **Intensity Accuracy** | 0.450 | — | 0.335 | — | — |
| Intensity F1 (macro) | 0.367 | — | 0.288 | — | — |

_Table 10: Fine-Tune Recovery (vs Zero-Shot)_

| Recovery | U-Net | FNO |
| --- | --- | --- |
| Direction Accuracy | −0.013 | **+0.098** |
| Intensity Accuracy | **+0.090** | −0.022 |

Fine-tuning results are mixed:
- **Intensity improves substantially** for U-Net (+9.0pp), suggesting intensity-relevant features are adaptable with limited data.
- **Direction is harder to recover**: U-Net's direction actually degrades slightly (−1.3pp) after fine-tuning, likely due to overfitting on only 354 SP samples.
- **FNO gains the most on direction** (+9.8pp) but from a lower baseline, suggesting its spectral features are more adaptable to new directional patterns.

_TODO: Insert figure — Fine-tuned confusion matrices for direction (`figures/comparison_cm_dir_finetuned.png`)_

_TODO: Insert figure — Radar chart comparing all metrics across models (`figures/comparison_radar.png`)_

### 4.4 Overfitting Analysis

With only ~3,250 training samples, overfitting is the dominant challenge:

_Table 11: Overfitting Severity_

| Model | Parameters | Params-per-Sample | Epochs Before Early Stop | Severity |
| --- | --- | --- | --- | --- |
| FNO | 10.3M | 3,170:1 | 18/80 | Severe |
| U-Net | 9.8M | 3,015:1 | 144/300 | Moderate |

**FNO overfits fastest** despite similar parameter count to U-Net. Spectral convolutions have high expressivity but lack the inductive biases (locality, skip connections) that help U-Net regularise. **U-Net trained longest** thanks to aggressive augmentation (Mixup, CutOut, channel dropout, EMA).

_TODO: Insert figure — Per-class F1 comparison (`figures/comparison_perclass_f1.png`)_

---

## Part 5: Explainability & Ablation

To understand what the models have learned and which features drive predictions, we conduct ablation studies and SHAP analysis using the baseline U-Net checkpoint. Full analysis is in `supplementary-notebooks/supplementary-analysis/feature-ablation-and-shap.ipynb`.

### 5.1 Modality Ablation

We evaluate six masking configurations to measure the contribution of each input modality. Each configuration zeros out one or more modalities while keeping the others intact:

_Table 12: Ablation Configurations_

| Configuration | Grid (15ch) | Env (40-d) | 1D (4-d) | Tests |
| --- | --- | --- | --- | --- |
| No Grid | ✗ | ✓ | ✓ | How much does removing spatial data hurt? |
| No Env | ✓ | ✗ | ✓ | How much does removing environmental context hurt? |
| No 1D | ✓ | ✓ | ✗ | How much does removing track data hurt? |
| Only Grid | ✓ | ✗ | ✗ | Can spatial data alone predict? |
| Only Env | ✗ | ✓ | ✗ | Can environmental features alone predict? |
| Only 1D | ✗ | ✗ | ✓ | Can track data alone predict? |

In [ ]:
# Modality ablation: zero-out masking functions
# (excerpt from supplementary-notebooks/supplementary-analysis/feature-ablation-and-shap.ipynb)

modality_configs = {
    "No Grid":  lambda g, e, d: (torch.zeros_like(g), e, d),
    "No Env":   lambda g, e, d: (g, torch.zeros_like(e), d),
    "No 1D":    lambda g, e, d: (g, e, torch.zeros_like(d)),
    "Only Grid": lambda g, e, d: (g, torch.zeros_like(e), torch.zeros_like(d)),
    "Only Env":  lambda g, e, d: (torch.zeros_like(g), e, torch.zeros_like(d)),
    "Only 1D":   lambda g, e, d: (torch.zeros_like(g), torch.zeros_like(e), d),
}

**Interpretation:** The ablation reveals the relative importance of each modality for direction vs intensity prediction. We also perform **leave-one-group-out** ablation within each modality (e.g., zeroing SST, u-wind, v-wind, geopotential, wind shear, or vorticity individually) to identify which specific physical variables drive each prediction task.

_TODO: Insert figure — Modality ablation bar chart showing accuracy drop per configuration_

### 5.2 SHAP Explainability

We use **SHAP GradientExplainer** [6] on the environmental features to quantify per-feature contributions to direction predictions. A wrapper class isolates the env input by fixing grid and 1D to their dataset means:

In [ ]:
# SHAP wrapper: isolates env features for attribution
# (excerpt from supplementary-notebooks/supplementary-analysis/feature-ablation-and-shap.ipynb)

class EnvWrapper(nn.Module):
    """Wraps the U-Net so SHAP only sees the env vector.
    Grid and d1d are fixed to their dataset means."""
    def __init__(self, model, fixed_grid, fixed_d1d):
        super().__init__()
        self.model = model
        self.register_buffer("fixed_grid", fixed_grid)
        self.register_buffer("fixed_d1d", fixed_d1d)

    def forward(self, env):
        B = env.shape[0]
        grid = self.fixed_grid.expand(B, -1, -1, -1)
        d1d = self.fixed_d1d.expand(B, -1)
        dir_logits, _ = self.model(grid, env, d1d)
        return dir_logits  # (B, 8) direction logits

**Interpretation:** By fixing the grid and 1D inputs to their means, we ensure that SHAP values reflect the marginal contribution of each environmental feature, without confounds from the spatial data. The analysis uses 100 background samples and 200 explanation samples, producing beeswarm plots and grouped mean |SHAP| bar charts that highlight which environmental variables (wind, shear, historical direction/intensity) are most important for each direction class.

We also compute **gradient-based grid attribution** (`mean(|∂loss/∂input|)` per channel) as a tractable alternative to full SHAP on the high-dimensional (15, 81, 81) grid tensors.

_TODO: Insert figure — SHAP beeswarm plot for environmental features_

_TODO: Insert figure — Gradient-based grid channel attribution bar chart_

---

## Part 6: Per-Storm Analysis

Beyond aggregate metrics, we analyse model predictions at the **individual storm level** to understand when and why models fail. Each model notebook includes a per-storm prediction timeline showing the sequence of predicted vs actual direction/intensity labels across a storm's lifetime.

_TODO: Insert figure — Example storm timeline showing predicted vs actual labels over time (`figures/unet_storm_timeline.png` or `figures/fno_v2_storm_timeline.png`)_

These timelines reveal patterns such as:
- Models tend to fail during **recurvature events** (when a storm changes direction sharply)
- **Rapid intensification** is consistently the hardest class to predict across all models
- Zero-shot predictions on SP storms show systematic biases that fine-tuning partially corrects

---

## Conclusion

This project investigated cross-basin generalisation for tropical cyclone forecasting, training models on the Western Pacific and evaluating transfer to the data-scarce South Pacific.

**Key findings:**

1. **U-Net is the strongest overall architecture** for this task. It achieves the highest direction accuracy both in-basin (0.625) and cross-basin (0.414), has the smallest transfer gap, and resists overfitting best thanks to aggressive augmentation.

2. **FNO offers complementary strengths.** Its spectral approach is inherently basin-agnostic and shows the largest fine-tuning recovery on direction (+9.8pp), suggesting Fourier features are more adaptable to new directional patterns.

3. **Temporal conditioning via FiLM** is a principled addition that allows models to adapt to storm lifecycle stage. Results pending for the improved variants (U-Net+FiLM, FNO v2, U-FNO).

4. **The transfer gap remains substantial.** Even the best zero-shot direction accuracy (0.414) is far from operational utility, highlighting the fundamental difficulty of cross-hemisphere generalisation.

5. **Explainability analysis** via ablation and SHAP confirms that physics-relevant features (wind shear, SST, vorticity) drive predictions, supporting the hypothesis that universal atmospheric physics enables cross-basin transfer.

All code, trained checkpoints, and analysis notebooks are available in this repository. For the most recent updates (post-March 18 meeting), see `README_implementation.md`.

---

## References

[1] Huang, C., Mu, P., Zhang, J., et al. (2025) 'Benchmark dataset and deep learning method for global tropical cyclone forecasting', *Nature Communications*, 16, 5923.

[2] DeMaria, M. and Kaplan, J. (1994) 'A Statistical Hurricane Intensity Prediction Scheme (SHIPS) for the Atlantic Basin', *Weather and Forecasting*, 9(2), pp. 209-220.

[3] Perez, E., Strub, F., de Vries, H., Dumoulin, V. and Courville, A. (2018) 'FiLM: Visual Reasoning with a General Conditioning Layer', *Proceedings of the AAAI Conference on Artificial Intelligence*.

[4] Li, Z., Kovachki, N., Azizzadenesheli, K., Liu, B., Bhattacharya, K., Stuart, A. and Anandkumar, A. (2021) 'Fourier Neural Operator for Parametric Partial Differential Equations', *International Conference on Learning Representations (ICLR)*.

[5] Wen, G., Li, Z., Azizzadenesheli, K., Anandkumar, A. and Benson, S.M. (2022) 'U-FNO: An enhanced Fourier neural operator-based deep-learning model for multiphase flow', *Advances in Water Resources*, 163, 104180.

[6] Lundberg, S.M. and Lee, S.-I. (2017) 'A Unified Approach to Interpreting Model Predictions', *Advances in Neural Information Processing Systems (NeurIPS)*, pp. 4765-4774.

[7] Qu, W., et al. (2025) 'Accurate tropical cyclone intensity forecasts using a non-iterative spatiotemporal transformer model', *npj Climate and Atmospheric Science*.

[8] Pathak, J., Subramanian, S., Harrington, P., et al. (2022) 'FourCastNet: A Global Data-driven High-resolution Weather Model using Adaptive Fourier Neural Operators', *arXiv preprint*, arXiv:2202.11214.

---

## Appendix: File Reference

_Table 13: Key Files in This Repository_

| File | Description | Status |
| --- | --- | --- |
| `README_implementation.md` | **Go-to document for all recent updates** (post-March 18 meeting) | Up to date |
| `supplementary-notebooks/preprocessing/data-preprocessing-pipeline.ipynb` | Shared data preprocessing pipeline | Complete |
| `supplementary-notebooks/preprocessing/temporal-feature-extraction.ipynb` | Temporal feature extraction notebook (with visualisation & sanity checks) | Complete |
| `supplementary-notebooks/preprocessing/temporal-feature-extraction.py` | Temporal feature extraction standalone script | Complete |
| `supplementary-notebooks/models/unet-baseline.ipynb` | Baseline U-Net | Trained |
| `supplementary-notebooks/models/unet-film-temporal.ipynb` | U-Net + FiLM time conditioning | Ready to train |
| `supplementary-notebooks/models/fno-baseline.ipynb` | Baseline FNO | Trained |
| `supplementary-notebooks/models/fno-v2-padded-film.ipynb` | FNO v2 (padding + 3 layers + FiLM) | Ready to train |
| `supplementary-notebooks/models/ufno-hybrid.ipynb` | U-FNO hybrid gated architecture | Ready to train |
| `supplementary-notebooks/models/resnet.ipynb` | ResNet-152 baseline | Trained (deprioritised) |
| `supplementary-notebooks/supplementary-analysis/feature-ablation-and-shap.ipynb` | Ablation + SHAP explainability | Ready to run |
| `supplementary-notebooks/supplementary-analysis/model-comparison.ipynb` | 6-model comparison with visualisations | Needs all checkpoints |